# C1 — Gaussian vs. Laplace NLL, at matched `beta`

Same machinery as `C0_v1_v2.ipynb`, with **one variable moved**: the `v1` NLL checkpoints are read from `models/v1/nll_beta/<arch>_nll/` (trained with `beta_gaussian_nll_loss`) instead of `models/v1/<arch>_nll/` (trained with plain `gaussian_nll_loss`).

**Why this notebook exists.** C0's NLL comparison moves two things at once: the *distribution* (Gaussian → Laplace) **and** the *beta reweighting* (`gaussian_nll_loss` applies none, i.e. `beta = 0`; `laplace_nll_loss` runs at `settings.NLL_BETA`). A delta measured there cannot be attributed to either. Holding `beta` at `settings.NLL_BETA` on both sides leaves the distribution as the only difference **inside the likelihood term**, which is what §8 is really asking about.

| | `v1` (this notebook) | `v2` (`deterministic/` + `nll/`) |
|---|---|---|
| normalization | `BatchNormalization` | `GroupNormalization` (`scripts/norm_utils.py`) |
| ReLU | `layers.ReLU()` | `_ReLUFix` (tensorflow-metal NaN workaround) |
| deterministic loss | per-architecture (`combined_loss` / `combined_loss_advanced`) | unified `combined_loss` (Charbonnier + 1-MS-SSIM) |
| NLL loss | `beta_nll` (`models/v1/nll_beta/`) — **beta-weighted Gaussian** | unified `laplace_nll` — **beta-weighted Laplace** |
| 2nd NLL channel | Gaussian **log-variance** → `sigma = exp(0.5 * ch1)` | Laplace **log-scale** → `sigma = sqrt(2) * exp(ch1)` |
| optimizer | plain Adam | Adam + `weight_decay` + `clipvalue` |

**Read the tables with these four caveats in mind.**

1. **This isolates the distribution, not the whole `v1 → v2` delta.** Normalization, the ReLU workaround and the optimizer (`weight_decay` + `clipvalue`) still differ between the two generations. A change seen here is attributable to *Gaussian vs. Laplace, or to any of those three* — it is simply no longer attributable to the beta weighting.
2. **Verify that `models/v1/nll_beta/` was actually trained at `settings.NLL_BETA = 0.5`.** If those checkpoints used a different beta, the confound is back and nothing below isolates anything. `logs/` is the place to check.
3. The NLL second channel means *different things* in the two generations (row 5 above), so §3 converts it per generation rather than applying one formula to both — the exact mistake `fixing.md` #10 documents. Everything derived from `sigma` (`\|z\|`, structural z, calibration) depends on that conversion being right.
4. `nll` in §8 is a **negative log-likelihood under a different distribution** on each side (Gaussian nats vs. Laplace nats) and is therefore *not* comparable across the two columns. Coverage, `z_std`, Spearman(error, sigma), sharpness and dispersion treat `sigma` as a plain standard deviation and *are* comparable.

**The four deterministic rows are identical to C0's** — nothing about the deterministic checkpoints or losses changes here. Only the four `*_nll` rows carry new information; the rest is kept so §5-§9 stay self-contained. Comment out the four `deterministic` entries in `ARCH_PAIRS` (§2) to roughly halve the runtime.

**What to look at first**: `dispersion` and `error_sigma_spearman` in §8. C0 showed both collapsing on the v2 side (dispersion 2.61 → 0.18 on `unet_nll`), i.e. a `sigma` that went nearly spatially constant. If they collapse against the beta-Gaussian baseline too, the cause is not the beta weighting; the remaining suspects are the distribution itself and the `weight_decay` / `clipvalue` added to the optimizer in v2, which apply to the `sigma` head as much as to the rest of the network.

Companion notebooks: `C0_v1_v2.ipynb` (the two generations exactly as they were trained), `040`/`041` (signals vs. ground-truth masks, current generation only), `052` (a user-chosen subset of current models against each other).

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [1]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check. Same building blocks as the 04x/05x series: `scripts.delta_analysis` for the delta/structural maps, `scripts.calibration` for the z-score signals and the sigma metrics, `scripts.detection` for AUROC against the hand-drawn masks, `scripts.stroke_stats` for the reference-free corroboration.

In [2]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.calibration import (
    evaluate_calibration,
    laplace_sigma_from_scale,
    learned_zscore,
    structural_zscore,
)
from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer import load_model
from scripts.trainer_nll import load_model_nll
from scripts.visualization_nll import DEFAULT_Z_SCALE, plot_signal_comparison

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. The `data/test/` images

Every `(rgb, ir)` pair under `data/test/`, discovered generically. The three with a hand-drawn mask in `data/test/annotations/*_Map.png` (`GT01`, `GT02`, `GT03`) additionally carry a detection ground truth — §6's AUROC is restricted to those, everything else (fidelity, coherence, calibration) needs no mask and runs on all of them.

In [3]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"

rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg")) + sorted(TEST_RGB_DIR.glob("*.png"))
image_pairs = [
    (p, TEST_IR_DIR / p.name)
    for p in sorted(rgb_paths)
    if (TEST_IR_DIR / p.name).exists()
]
gt_stems = {
    p.name.removesuffix("_Map.png") for p in sorted(ANNOTATIONS_DIR.glob("*_Map.png"))
}
gt_stems &= {p.stem for p, _ in image_pairs}

print(f"data/test/ images: {len(image_pairs)} | {[p.stem for p, _ in image_pairs]}")
print(f"with a ground-truth mask: {sorted(gt_stems)}")

if not image_pairs:
    raise RuntimeError("No (rgb, ir) pairs found under data/test/.")

data/test/ images: 10 | ['GT01', 'GT02', 'GT03', 'bridge', 'case', 'face', 'green', 'modern', 'modern2', 'total']
with a ground-truth mask: ['GT01', 'GT02', 'GT03']


## 2. The two generations, and which architectures they share

`VERSIONS` says where each generation's checkpoints live and how to read its NLL second channel — the only two things that differ structurally between them. Unlike C0, the `v1` NLL entry points at `models/v1/nll_beta/`, so both sides are beta-weighted at `settings.NLL_BETA`; `load_version` already passes that beta to `load_model_nll` for either generation.

`sigma_source` stays `"gaussian"`: those checkpoints still carry a Gaussian log-variance in channel 1. Only the loss they were trained with changes, not how the channel is read.

`ARCH_PAIRS` lists only architectures present in **both** generations, since a one-sided entry has nothing to compare against: `unet_v2`/`unet_restormer` are v1-only, `efficientnet_unet_ft`/`efficientnet_unet_nll_ft` and `unet_dilated`/`unet_v2_dilated` are v2-only. Any architecture missing a checkpoint on either side is skipped with a message rather than raising.

To go back to C0's pairing — plain `gaussian_nll` (`beta = 0`) on the v1 side — point `VERSIONS["v1"]["dir"]["nll"]` back at `settings.MODELS_DIR / "v1"` and set `nll_loss` to `"gaussian_nll"`.

In [4]:
VERSIONS = {
    "v1": {
        "dir": {
            "deterministic": settings.MODELS_DIR / "v1",
            "nll": settings.MODELS_DIR / "v1" / "nll_beta",
        },
        # Pre-Round-2: the NLL head was trained as a Gaussian, so channel 1
        # is a log-variance and the NLL metric in §8 is in Gaussian nats.
        # Unlike C0, these are the `nll_beta` siblings — beta-weighted at
        # settings.NLL_BETA, matching v2's laplace_nll, so the beta weighting
        # is held fixed and only the distribution differs.
        "nll_loss": "beta_nll",
        "sigma_source": "gaussian",
    },
    "v2": {
        "dir": {
            "deterministic": settings.MODELS_DIR / "deterministic",
            "nll": settings.MODELS_DIR / "nll",
        },
        # fixing.md #10: every NLL architecture is now a Laplace, so
        # channel 1 is a log-scale and sigma = b * sqrt(2).
        "nll_loss": "laplace_nll",
        "sigma_source": "laplace",
    },
}

ARCH_PAIRS = [
    ("unet", "deterministic"),
    ("resunet", "deterministic"),
    ("attention_unet", "deterministic"),
    ("efficientnet_unet", "deterministic"),
    ("unet_nll", "nll"),
    ("resunet_nll", "nll"),
    ("attention_unet_nll", "nll"),
    ("efficientnet_unet_nll", "nll"),
]


def load_version(arch: str, family: str, version: str) -> tf.keras.Model:
    """Load one architecture's checkpoint from one generation."""
    spec = VERSIONS[version]
    model_dir = spec["dir"][family]
    if family == "deterministic":
        return load_model(arch, model_dir=model_dir)
    return load_model_nll(
        arch, model_dir=model_dir, loss_name=spec["nll_loss"], beta=settings.NLL_BETA
    )


missing = [
    (arch, version)
    for arch, family in ARCH_PAIRS
    for version in VERSIONS
    if not (VERSIONS[version]["dir"][family] / arch / "best_model.keras").exists()
]
print(f"Architecture pairs to compare: {[a for a, _ in ARCH_PAIRS]}")
print(f"Missing checkpoints (will be skipped): {missing or 'none'}")

Architecture pairs to compare: ['unet', 'resunet', 'attention_unet', 'efficientnet_unet', 'unet_nll', 'resunet_nll', 'attention_unet_nll', 'efficientnet_unet_nll']
Missing checkpoints (will be skipped): none


## 3. Signals, per generation

Identical construction to `040`/`041`/`052` — **raw delta** `|real_IR - mu|`, **structural delta** `1 - local SSIM structure`, **confidence** (their normalized agreement) for every model; plus **`\|z\|`** and **structural z** for the NLL architectures, which need a `sigma`.

The one thing that is *not* identical across the two generations is that `sigma`. `sigma_from_channel` dispatches on `VERSIONS[version]["sigma_source"]`: `exp(0.5 * ch1)` for v1's Gaussian log-variance, `laplace_sigma_from_scale(exp(ch1))` (i.e. `sqrt(2) * b`) for v2's Laplace log-scale. Applying either formula to both generations would make the z-signals and every calibration number below silently wrong — this is `fixing.md` #10's mistake, and the reason this dispatch exists rather than one shared line.

In [5]:
MASK_THRESHOLD = 127  # midpoint threshold for the mask's anti-aliased edges

SIGNAL_KINDS = ("raw delta", "structural delta", "confidence")
NLL_SIGNAL_KINDS = ("|z| (raw/sigma)", "structural z")


def load_pair(rgb_path: Path, ir_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load an ``(rgb, ir)`` pair as float arrays in ``[0, 1]``."""
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    return rgb, ir


def load_mask(stem: str) -> np.ndarray:
    """Load a hand-drawn ground-truth mask as a boolean array."""
    mask_path = ANNOTATIONS_DIR / f"{stem}_Map.png"
    return np.array(Image.open(mask_path).convert("L")) > MASK_THRESHOLD


def sigma_from_channel(log_channel: np.ndarray, sigma_source: str) -> np.ndarray:
    """Turn an NLL head's second channel into a standard deviation."""
    if sigma_source == "gaussian":
        return np.exp(0.5 * log_channel)
    if sigma_source == "laplace":
        return laplace_sigma_from_scale(np.exp(log_channel))
    raise ValueError(f"Unknown sigma_source '{sigma_source}'.")


def fidelity(ir: np.ndarray, mu: np.ndarray) -> dict[str, float]:
    """MAE/SSIM/PSNR of a prediction against the real IR, as in 030."""
    real = tf.constant(ir[..., np.newaxis])
    pred = tf.constant(mu[..., np.newaxis])
    return {
        "mae": float(np.mean(np.abs(ir - mu))),
        "ssim": float(tf.image.ssim(real, pred, max_val=1.0)),
        "psnr": float(tf.image.psnr(real, pred, max_val=1.0)),
    }


def predict_signals(
    model: tf.keras.Model, family: str, version: str, rgb: np.ndarray, ir: np.ndarray
) -> tuple[dict[str, float], dict[str, np.ndarray], np.ndarray | None, np.ndarray]:
    """Predict one image and build every signal available to this family."""
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = ir.shape
    pred = model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, :]

    mu = pred[..., 0]
    result = analyze_delta(ir, mu)
    signals = {
        "raw delta": result.raw_delta,
        "structural delta": result.structural_delta,
        "confidence": result.confidence_map,
    }

    sigma = None
    if family == "nll":
        sigma = sigma_from_channel(pred[..., 1], VERSIONS[version]["sigma_source"])
        signals["|z| (raw/sigma)"] = np.abs(learned_zscore(ir, mu, sigma))
        signals["structural z"] = structural_zscore(result.structural_delta, sigma)

    return fidelity(ir, mu), signals, sigma, mu

## 4. The sweep

One architecture at a time, both generations held in memory together (so their signals are built from the exact same image arrays), every `data/test/` image scored, then both released before moving to the next architecture — v1 checkpoints are ~380 MB each, so loading all sixteen at once would not fit.

`PLOT_IMAGES` and `PLOT_SIGNALS` control only the *figures*; the numeric tables in §5-§8 always cover every image and every signal. The defaults plot the three ground-truth images, where a visible difference can be checked against the mask — widen to `{p.stem for p, _ in image_pairs}` for the full qualitative sweep.

This is the long cell: `len(ARCH_PAIRS) x len(image_pairs) x 2` full-image predictions on images up to ~10 Mpx, plus the Gaussian-windowed SSIM decomposition for each.

In [6]:
PLOT_IMAGES = set()# = set(gt_stems)  # -> {p.stem for p, _ in image_pairs} for all of them
PLOT_SIGNALS = set()# ("raw delta", "structural delta")

fidelity_rows: dict[tuple[str, str], dict[str, list[float]]] = {}
coherence_rows: dict[tuple[str, str, str], list[float]] = {}
auroc_rows: dict[tuple[str, str, str], dict[str, float]] = {}
ap_rows: dict[tuple[str, str, str], dict[str, float]] = {}
calibration_rows: dict[tuple[str, str], dict[str, list[float]]] = {}

compared_archs: list[tuple[str, str]] = []

for arch, family in ARCH_PAIRS:
    loaded: dict[str, tf.keras.Model] = {}
    for version in VERSIONS:
        try:
            loaded[version] = load_version(arch, family, version)
        except FileNotFoundError as exc:
            print(f"[skip] {arch} @ {version}: {exc}")

    if len(loaded) < len(VERSIONS):
        print(f"[skip] {arch}: needs a checkpoint in every generation to compare")
        del loaded
        gc.collect()
        tf.keras.backend.clear_session()
        continue

    compared_archs.append((arch, family))
    print(f"\n=== {arch} ({family}) — {' vs. '.join(loaded)} ===")

    for rgb_path, ir_path in image_pairs:
        stem = rgb_path.stem
        rgb, ir = load_pair(rgb_path, ir_path)
        mask = load_mask(stem) if stem in gt_stems else None

        per_version = {}
        for version, model in loaded.items():
            scores, signals, sigma, mu = predict_signals(
                model, family, version, rgb, ir
            )
            per_version[version] = signals

            for metric, value in scores.items():
                fidelity_rows.setdefault((arch, version), {}).setdefault(
                    metric, []
                ).append(value)

            for kind, signal in signals.items():
                coherence_rows.setdefault((arch, version, kind), []).append(
                    stroke_coherence(signal).coherence
                )
                if mask is not None:
                    detection = evaluate_detection(signal, mask)
                    auroc_rows.setdefault((arch, version, kind), {})[stem] = (
                        detection.auroc
                    )
                    ap_rows.setdefault((arch, version, kind), {})[stem] = (
                        detection.average_precision
                    )

            if sigma is not None:
                distribution = VERSIONS[version]["sigma_source"]
                summary = evaluate_calibration(
                    ir, mu, sigma, distribution=distribution
                ).summary()
                for metric, value in summary.items():
                    calibration_rows.setdefault((arch, version), {}).setdefault(
                        metric, []
                    ).append(value)

        if stem in PLOT_IMAGES:
            for kind in PLOT_SIGNALS:
                fig = plot_signal_comparison(
                    ir,
                    {v: s[kind] for v, s in per_version.items()},
                    title=f"{arch} — {kind} — {stem}",
                    vrange=(0.0, 1.0),
                )
                plt.show()
                plt.close(fig)
            if family == "nll":
                for kind in NLL_SIGNAL_KINDS:
                    scaled, vrange = DEFAULT_Z_SCALE.apply_many(
                        {v: s[kind] for v, s in per_version.items()}
                    )
                    fig = plot_signal_comparison(
                        ir, scaled, title=f"{arch} — {kind} — {stem}", vrange=vrange
                    )
                    plt.show()
                    plt.close(fig)

        print(f"  {stem}: scored")
        del per_version, rgb, ir, mask

    del loaded
    gc.collect()
    tf.keras.backend.clear_session()

print(f"\nCompared: {[a for a, _ in compared_archs]}")
if not compared_archs:
    raise RuntimeError(
        "No architecture had a checkpoint in both generations — nothing to compare."
    )

2026-08-26 23:43:56.780746: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-08-26 23:43:56.780784: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-08-26 23:43:56.780792: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-08-26 23:43:56.780813: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-26 23:43:56.780831: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



=== unet (deterministic) — v1 vs. v2 ===


2026-08-26 23:44:00.152257: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  GT01: scored
  GT02: scored
  GT03: scored
  bridge: scored
  case: scored
  face: scored
  green: scored
  modern: scored
  modern2: scored
  total: scored

=== resunet (deterministic) — v1 vs. v2 ===
  GT01: scored
  GT02: scored
  GT03: scored
  bridge: scored
  case: scored
  face: scored
  green: scored
  modern: scored
  modern2: scored
  total: scored

=== attention_unet (deterministic) — v1 vs. v2 ===
  GT01: scored
  GT02: scored
  GT03: scored
  bridge: scored
  case: scored
  face: scored
  green: scored
  modern: scored
  modern2: scored
  total: scored

=== efficientnet_unet (deterministic) — v1 vs. v2 ===
  GT01: scored
  GT02: scored
  GT03: scored
  bridge: scored
  case: scored
  face: scored
  green: scored
  modern: scored
  modern2: scored
  total: scored

=== unet_nll (nll) — v1 vs. v2 ===
  GT01: scored
  GT02: scored
  GT03: scored
  bridge: scored
  case: scored
  face: scored
  green: scored
  modern: scored
  modern2: scored
  total: scored

=== resunet_nll 

## 5. Reconstruction fidelity — how well each generation predicts the IR

`mae`/`ssim`/`psnr` of `mu` against the real IR, averaged over all `data/test/` images. This is the same axis `030_evaluation.ipynb` reports, and it measures the *forward* task only: a model can improve here and still surface the underdrawing worse, which is what §6 and §7 are for.

`gain` is always oriented so that **positive means v2 improved**: `v2 - v1` for `ssim`/`psnr`, `v1 - v2` for `mae` (where lower is better). Read a negative `gain` as a regression, whatever the metric.

In [7]:
# True where a *lower* value is the better one, so `gain` below can be
# oriented consistently: positive always means v2 improved.
LOWER_IS_BETTER = {"mae": True, "ssim": False, "psnr": False}


def mean_of(rows: dict, key, metric: str) -> float:
    return float(np.mean(rows[key][metric]))


def gain(metric: str, v1: float, v2: float) -> float:
    return v1 - v2 if LOWER_IS_BETTER[metric] else v2 - v1


col_w = 26
header = "architecture".ljust(col_w)
for metric in LOWER_IS_BETTER:
    header += (
        f"{metric} v1".rjust(12) + f"{metric} v2".rjust(12) + f"{metric} gain".rjust(13)
    )
print(header)
print("-" * len(header))

for arch, _ in compared_archs:
    row = arch.ljust(col_w)
    for metric in LOWER_IS_BETTER:
        v1 = mean_of(fidelity_rows, (arch, "v1"), metric)
        v2 = mean_of(fidelity_rows, (arch, "v2"), metric)
        row += f"{v1:.4f}".rjust(12) + f"{v2:.4f}".rjust(12)
        row += f"{gain(metric, v1, v2):+.4f}".rjust(13)
    print(row)

print(f"\n(mean over {len(image_pairs)} data/test/ images; gain > 0 means v2 better)")

architecture                    mae v1      mae v2     mae gain     ssim v1     ssim v2    ssim gain     psnr v1     psnr v2    psnr gain
-----------------------------------------------------------------------------------------------------------------------------------------
unet                            0.1048      0.1265      -0.0217      0.5899      0.6071      +0.0172     17.7655     16.6486      -1.1169
resunet                         0.1941      0.1084      +0.0857      0.5803      0.6120      +0.0317     13.1656     17.6901      +4.5246
attention_unet                  0.2095      0.1244      +0.0851      0.5753      0.6209      +0.0457     12.5419     16.7330      +4.1911
efficientnet_unet               0.1222      0.1284      -0.0062      0.2302      0.6051      +0.3749     16.6153     16.5362      -0.0792
unet_nll                        0.1203      0.1012      +0.0191      0.4949      0.5461      +0.0512     16.8855     18.1603      +1.2748
resunet_nll                     0.

## 6. Detection — the signal that actually matters

AUROC of each signal against the hand-drawn masks, averaged over the ground-truth images only (`GT01`/`GT02`/`GT03`). `0.5` is chance. With `n = 3` the mean hides a lot, so the per-image breakdown follows — read it before treating any `delta` here as a result.

This is the primary axis of the whole comparison: the project's deliverable is the residual that reveals the underdrawing, not the IR prediction itself.

In [8]:
def detection_table(
    rows: dict[tuple[str, str, str], dict[str, float]], label: str
) -> None:
    kinds_seen = {kind for _, _, kind in rows}
    ordered_kinds = [k for k in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS) if k in kinds_seen]
    name_w = 44
    print("signal".ljust(name_w) + "v1".rjust(10) + "v2".rjust(10) + "delta".rjust(12))
    print("-" * (name_w + 32))
    for arch, _ in compared_archs:
        for kind in ordered_kinds:
            if (arch, "v1", kind) not in rows or (arch, "v2", kind) not in rows:
                continue
            v1 = float(np.mean(list(rows[(arch, "v1", kind)].values())))
            v2 = float(np.mean(list(rows[(arch, "v2", kind)].values())))
            name = f"{arch} [{kind}]"
            print(
                name.ljust(name_w)
                + f"{v1:.4f}".rjust(10)
                + f"{v2:.4f}".rjust(10)
                + f"{v2 - v1:+.4f}".rjust(12)
            )
    print(f"\n({label}, mean over {sorted(gt_stems)})")


if auroc_rows:
    print("AUROC\n")
    detection_table(auroc_rows, "AUROC")
    print("\n\naverage precision\n")
    detection_table(ap_rows, "average precision")
else:
    print("No ground-truth mask found under data/test/annotations/ — section skipped.")

AUROC

signal                                              v1        v2       delta
----------------------------------------------------------------------------
unet [raw delta]                                0.5763    0.4726     -0.1037
unet [structural delta]                         0.6461    0.7073     +0.0612
unet [confidence]                               0.4162    0.5193     +0.1031
resunet [raw delta]                             0.4075    0.5520     +0.1445
resunet [structural delta]                      0.7325    0.6810     -0.0515
resunet [confidence]                            0.6046    0.4596     -0.1450
attention_unet [raw delta]                      0.4032    0.4993     +0.0961
attention_unet [structural delta]               0.7202    0.7007     -0.0195
attention_unet [confidence]                     0.6158    0.5114     -0.1044
efficientnet_unet [raw delta]                   0.5000    0.4810     -0.0190
efficientnet_unet [structural delta]            0.6685    0.6856     

### Per-image AUROC breakdown

The `n = 3` mean above can be carried entirely by one favourable image. Every ground-truth image is shown here for both generations, so a `delta` that only exists on one painting is visible as such.

In [9]:
if auroc_rows:
    stems = sorted(gt_stems)
    name_w = 44
    header = "signal".ljust(name_w) + "".join(
        f"{stem} v1".rjust(12) + f"{stem} v2".rjust(12) for stem in stems
    )
    print(header)
    print("-" * len(header))
    for arch, _ in compared_archs:
        for kind in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS):
            if (arch, "v1", kind) not in auroc_rows:
                continue
            row = f"{arch} [{kind}]".ljust(name_w)
            for stem in stems:
                for version in ("v1", "v2"):
                    row += f"{auroc_rows[(arch, version, kind)][stem]:.3f}".rjust(12)
            print(row)
else:
    print("No ground-truth mask found — section skipped.")

signal                                           GT01 v1     GT01 v2     GT02 v1     GT02 v2     GT03 v1     GT03 v2
--------------------------------------------------------------------------------------------------------------------
unet [raw delta]                                   0.714       0.442       0.540       0.461       0.474       0.515
unet [structural delta]                            0.688       0.815       0.597       0.662       0.653       0.645
unet [confidence]                                  0.327       0.509       0.511       0.488       0.410       0.561
resunet [raw delta]                                0.251       0.612       0.445       0.536       0.527       0.508
resunet [structural delta]                         0.850       0.784       0.659       0.628       0.688       0.632
resunet [confidence]                               0.722       0.358       0.591       0.474       0.501       0.547
attention_unet [raw delta]                         0.212       0

## 7. Stroke coherence — the reference-free corroboration

How much of each signal is oriented, line-like structure rather than isotropic noise (`scripts.stroke_stats`). It needs no mask, so unlike §6 it runs on **all** `data/test/` images and is a structurally independent check on the AUROC verdict: a generation that improves detection AUROC on three paintings but degrades coherence across ten is not obviously better.

In [10]:
name_w = 44
print("signal".ljust(name_w) + "v1".rjust(16) + "v2".rjust(16) + "delta".rjust(12))
print("-" * (name_w + 44))
for arch, _ in compared_archs:
    for kind in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS):
        if (arch, "v1", kind) not in coherence_rows:
            continue
        v1 = np.array(coherence_rows[(arch, "v1", kind)])
        v2 = np.array(coherence_rows[(arch, "v2", kind)])
        print(
            f"{arch} [{kind}]".ljust(name_w)
            + f"{v1.mean():.3f}±{v1.std():.3f}".rjust(16)
            + f"{v2.mean():.3f}±{v2.std():.3f}".rjust(16)
            + f"{v2.mean() - v1.mean():+.4f}".rjust(12)
        )
print(f"\n(mean ± std over {len(image_pairs)} data/test/ images)")

signal                                                    v1              v2       delta
----------------------------------------------------------------------------------------
unet [raw delta]                                 0.251±0.069     0.244±0.069     -0.0074
unet [structural delta]                          0.332±0.054     0.357±0.065     +0.0247
unet [confidence]                                0.338±0.065     0.338±0.069     -0.0008
resunet [raw delta]                              0.268±0.064     0.244±0.068     -0.0239
resunet [structural delta]                       0.374±0.090     0.351±0.062     -0.0228
resunet [confidence]                             0.341±0.084     0.348±0.071     +0.0072
attention_unet [raw delta]                       0.274±0.066     0.253±0.079     -0.0217
attention_unet [structural delta]                0.415±0.101     0.373±0.071     -0.0425
attention_unet [confidence]                      0.353±0.086     0.342±0.079     -0.0107
efficientnet_unet [ra

## 8. Sigma calibration — NLL architectures only

`mae`/`ssim`/`psnr` and the AUROC of the delta signals all read `mu`; this section scores `sigma` on its own (`scripts.calibration`), averaged over all `data/test/` images — no mask needed, so the full set is used.

**`nll` is not comparable across the two columns**: v1's is a Gaussian NLL, v2's a Laplace NLL, two different densities in nats. It is printed because it is the number each generation was actually trained to minimise, not so the two can be subtracted — its `delta` column is deliberately blank.

The rest are distribution-agnostic and *are* comparable: `coverage_1s`/`coverage_2s` against nominal `0.6827`/`0.9545`, `z_std` (`1.0` is calibrated, `> 1` overconfident), `ence` (lower is better), `error_sigma_spearman` (higher: sigma tracks where the model is actually wrong), `sharpness` (mean sigma) and `dispersion` (its coefficient of variation — a large *constant* sigma would score well on coverage and be useless, which dispersion catches).

In [11]:
CALIB_KEYS = [
    "nll",
    "ence",
    "z_std",
    "coverage_1s",
    "coverage_2s",
    "error_sigma_spearman",
    "sharpness",
    "dispersion",
]
NOT_COMPARABLE = {"nll"}  # different distribution per generation

nll_archs = [arch for arch, family in compared_archs if family == "nll"]

if not nll_archs:
    print("No NLL architecture compared — section skipped.")
else:
    name_w = 24
    for arch in nll_archs:
        print(f"\n=== {arch} ===")
        print(
            "metric".ljust(name_w) + "v1".rjust(12) + "v2".rjust(12) + "delta".rjust(12)
        )
        print("-" * (name_w + 36))
        for metric in CALIB_KEYS:
            v1 = float(np.mean(calibration_rows[(arch, "v1")][metric]))
            v2 = float(np.mean(calibration_rows[(arch, "v2")][metric]))
            delta = "n/a" if metric in NOT_COMPARABLE else f"{v2 - v1:+.4f}"
            print(
                metric.ljust(name_w)
                + f"{v1:.4f}".rjust(12)
                + f"{v2:.4f}".rjust(12)
                + delta.rjust(12)
            )
    print("\nnominal coverage: 1s = 0.6827, 2s = 0.9545; calibrated z_std = 1.0")
    print("'n/a' = Gaussian nats (v1) vs. Laplace nats (v2), not subtractable")


=== unet_nll ===
metric                            v1          v2       delta
------------------------------------------------------------
nll                           0.1653     -0.5871         n/a
ence                          0.6726      0.2397     -0.4329
z_std                         0.3031      0.9090     +0.6059
coverage_1s                   0.9956      0.7058     -0.2899
coverage_2s                   1.0000      0.9525     -0.0475
error_sigma_spearman         -0.2116      0.1575     +0.3692
sharpness                     0.4461      0.1292     -0.3169
dispersion                    0.1058      0.1775     +0.0717

=== resunet_nll ===
metric                            v1          v2       delta
------------------------------------------------------------
nll                           0.3243     -0.4144         n/a
ence                          0.6015      0.2441     -0.3574
z_std                         0.3480      0.8073     +0.4592
coverage_1s                   0.9987      0.68

## 9. Verdict — where v2 wins and where it does not

A count, per architecture, of how many measured quantities improved. Deliberately coarse: it collapses three different axes (fidelity, detection, coherence) that do not have to agree, and detection rests on three masks. Use it to see *which architectures moved at all*, then read §5-§7 for what actually moved.

In [12]:
def tally(gains: list[float]) -> str:
    """``"k/n"`` — how many of these gains are improvements for v2."""
    if not gains:
        return "-"
    return f"{sum(g > 0 for g in gains)}/{len(gains)}"


ALL_KINDS = (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS)

name_w = 26
print(
    "architecture".ljust(name_w)
    + "fidelity".rjust(12)
    + "detection".rjust(12)
    + "coherence".rjust(12)
)
print("-" * (name_w + 36))

for arch, _ in compared_archs:
    fidelity_gains = [
        gain(
            metric,
            mean_of(fidelity_rows, (arch, "v1"), metric),
            mean_of(fidelity_rows, (arch, "v2"), metric),
        )
        for metric in LOWER_IS_BETTER
    ]
    detection_gains = [
        float(np.mean(list(auroc_rows[(arch, "v2", kind)].values())))
        - float(np.mean(list(auroc_rows[(arch, "v1", kind)].values())))
        for kind in ALL_KINDS
        if (arch, "v1", kind) in auroc_rows
    ]
    coherence_gains = [
        float(np.mean(coherence_rows[(arch, "v2", kind)]))
        - float(np.mean(coherence_rows[(arch, "v1", kind)]))
        for kind in ALL_KINDS
        if (arch, "v1", kind) in coherence_rows
    ]

    print(
        arch.ljust(name_w)
        + tally(fidelity_gains).rjust(12)
        + tally(detection_gains).rjust(12)
        + tally(coherence_gains).rjust(12)
    )

print("\n(v2 better / total measured; fidelity = mae, ssim, psnr)")

architecture                  fidelity   detection   coherence
--------------------------------------------------------------
unet                               1/3         2/3         1/3
resunet                            3/3         1/3         1/3
attention_unet                     3/3         1/3         0/3
efficientnet_unet                  1/3         2/3         3/3
unet_nll                           3/3         3/5         0/5
resunet_nll                        3/3         3/5         3/5
attention_unet_nll                 3/3         2/5         0/5
efficientnet_unet_nll              0/3         1/5         1/5

(v2 better / total measured; fidelity = mae, ssim, psnr)
